Soma total de vendas por dia, ordens com status = "Delivered" e state = "NY"

In [0]:
%python
bronze_path   = 'bikestore.bronze'
silver_path   = 'bikestore.silver'
gold_path     = 'bikestore.gold'
resource_path = "/Volumes/bikestore/resource/origem"
resource_path_volume = '/Volumes/bikestore/resource/origem'

In [0]:
%python
# criar varias tabelas temporárias de forma prática

silver_map = {
    # View_name --------------- path
    "tmp_silver_customers": f"{silver_path}.customers",
    "tmp_silver_orders": f"{silver_path}.orders",
    "tmp_silver_products": f"{silver_path}.product"
    }

for view_name, path in silver_map.items():
    (spark.table(path).createOrReplaceTempView(view_name))

In [0]:
select * from tmp_silver_customers limit 4

In [0]:
select * from tmp_silver_orders limit 4

In [0]:
select * from tmp_silver_products limit 4

Vendas por dia AND order_status = "Delivered" AND state = "NY"

In [0]:
select
  shipped_date,
  round(sum(total_sale), 2) as total_sale,
  source_orders_file,
  source_store_file,
  source_staffs_file,
  source_order_items_file
-- order_status,
-- state
from
  tmp_silver_orders
where
  1 = 1
  AND order_status = "Delivered"
  AND state = "NY"
  AND shipped_date IS NOT NULL
group by
  shipped_date,
  source_orders_file,
  source_store_file,
  source_staffs_file,
  source_order_items_file
-- shipped_date, state

In [0]:
%python
from pyspark.sql.functions import current_timestamp, lit

current_ts = current_timestamp()

df_sales_gold = spark.sql("""
    select
        shipped_date,
        round(sum(total_sale), 2) as total_sale,
        source_orders_file,
        source_store_file,
        source_staffs_file,
        source_order_items_file
        -- order_status,
        -- state
    from
         tmp_silver_orders
    where
        1 = 1
        AND order_status = "Delivered"
        AND state = "NY"
        AND shipped_date IS NOT NULL
    group by
        shipped_date,
        source_orders_file,
        source_store_file,
        source_staffs_file,
        source_order_items_file
-- shipped_date, state
""")\
    .withColumn("ingestion_timestamp", current_ts) \
    .withColumn("created_at", current_ts) \
    .withColumn("updated_at", lit(None).cast("timestamp"))

In [0]:
%python
display(df_sales_gold)

In [0]:
%python
#salvar em parquet como delta na camada silver


df_sales_gold.write\
    .mode("overwrite")\
    .format("delta")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"{gold_path}.sales_ny")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bikestore.logistics.sales_ny;

In [0]:
%sql
CREATE OR REPLACE TABLE bikestore.logistics.sales_ny AS
SELECT * FROM bikestore.gold.sales_ny;

In [0]:
%sql
select * from bikestore.logistics.sales_ny;